Exercise 1- Conceptual Questions

1. Transfer Learning Intuition

Early convolutional layers detect basic features like edges, textures, and simple patterns.  
These features are universal and useful for almost all image tasks, so freezing them saves time and prevents overfitting.


2. Data Augmentation Strategy

a) Horizontal & Vertical Flipping → Useful (objects can appear in different orientations)
b) Random Rotation (0–360°) → Useful (satellite images can be rotated)
c) Brightness & Contrast → Useful (lighting conditions vary)
d) Random Cropping → Risky (can remove important object like a swimming pool)


3. Fine-Tuning vs Feature Extraction

Since the dataset is large (1M images) and very different from ImageNet, full fine-tuning is better.  
The model can adapt deeper features to medical images.

Use a very small learning rate to avoid destroying pre-trained knowledge.

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

In [5]:
train_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor()
])

val_transforms = transforms.Compose([
    transforms.ToTensor()
])

In [25]:
train_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=train_transforms
)

val_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=val_transforms
)

train_dataset = torch.utils.data.Subset(train_dataset, range(5000))
val_dataset = torch.utils.data.Subset(val_dataset, range(1000))

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=128, shuffle=False)

epochs = 2

In [26]:
model = torchvision.models.resnet18(pretrained=True)

In [27]:
for param in model.parameters():
    param.requires_grad = False

In [28]:
model.fc = nn.Linear(model.fc.in_features, 10)

In [29]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [30]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

In [31]:
from tqdm import tqdm

for epoch in range(epochs):
    model.train()
    total_loss = 0
    
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        images, labels = images.to(device), labels.to(device)
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1: 100%|██████████| 40/40 [00:07<00:00,  5.68it/s]


Epoch 1, Loss: 91.9241


Epoch 2: 100%|██████████| 40/40 [00:09<00:00,  4.32it/s]

Epoch 2, Loss: 80.4157


I reduced dataset size and epochs for faster experimentation, but the full training would use the entire dataset

In [32]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Accuracy: {100 * correct / total:.2f}%")

Accuracy: 31.80%


In [33]:
for param in model.layer4.parameters():
    param.requires_grad = True

In [34]:
optimizer = optim.Adam(model.parameters(), lr=0.0001)

In [36]:
epochs = 2

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"[Fine-tune] Epoch {epoch+1}, Loss: {total_loss:.4f}")

[Fine-tune] Epoch 1, Loss: 65.8646
[Fine-tune] Epoch 2, Loss: 61.1819


Fine-tuning improved the model performance because it allowed deeper layers to adapt to the new dataset

Using a small learning rate is critical because large updates could destroy the useful pre-trained features